# Notebook 03 — Listas, Diccionarios, Bucles `for`, Funciones e Intro a NumPy
**Pungo Academy · Python y Ciencia de Datos Industrial desde Cero en Google Colab**

## Objetivo
Escalar de una variable individual a **cientos de sensores y múltiples equipos** sin repetir código:
1. Listas (`list`) para secuencias temporales e indexado (`[0]`, `[-1]`, slicing).
2. Diccionarios (`dict`) para fichas técnicas estructuradas de equipos.
3. Bucles `for` para auditar automáticamente toda una planta.
4. Funciones reutilizables (`def`) con parámetros y `return`.
5. Introducción a **NumPy** (`np.array`) para cálculos vectorizados instantáneos.
6. **Error Lab 03**: `IndexError` y `KeyError` + retos **Tu Turno**.

In [1]:
# 1. Listas: secuencia horaria de temperatura en la caldera CAL-01 (°C)
temps_caldera = [181.2, 183.5, 185.0, 188.4, 192.1, 194.8, 190.2, 186.7]

print("Total de lecturas:", len(temps_caldera))
print("Primera lectura [0]:", temps_caldera[0], "°C")
print("Última lectura [-1]:", temps_caldera[-1], "°C")
print("Primeras 3 horas [0:3]:", temps_caldera[0:3])
print(f"Mínima: {min(temps_caldera):.1f} °C | Máxima: {max(temps_caldera):.1f} °C")
promedio_temp = sum(temps_caldera) / len(temps_caldera)
print(f"Temperatura promedio del turno: {promedio_temp:.2f} °C")

Total de lecturas: 8
Primera lectura [0]: 181.2 °C
Última lectura [-1]: 186.7 °C
Primeras 3 horas [0:3]: [181.2, 183.5, 185.0]
Mínima: 181.2 °C | Máxima: 194.8 °C
Temperatura promedio del turno: 187.74 °C


## Paso 2 — Diccionarios: Fichas Técnicas de Activos
Una lista guarda datos por posición (`[0]`, `[1]`). Un **diccionario** guarda datos por nombre clave (`"equipo"`, `"vibracion"`), igual que una ficha técnica JSON o un registro SCADA.

In [2]:
# 2. Catálogo de 4 equipos de planta usando una lista de diccionarios
equipos_planta = [
    {"tag": "CAL-01", "nombre": "Caldera Principal",     "temp_c": 188.5, "vib_mms": 2.1, "limite_vib": 3.5},
    {"tag": "TUR-01", "nombre": "Turbina de Vapor",      "temp_c": 318.2, "vib_mms": 4.4, "limite_vib": 4.0},
    {"tag": "BOM-02", "nombre": "Bomba de Alimentación", "temp_c": 65.4,  "vib_mms": 2.6, "limite_vib": 3.5},
    {"tag": "COM-01", "nombre": "Compresor de Aire",     "temp_c": 99.1,  "vib_mms": 4.7, "limite_vib": 4.2},
]

print("Primer equipo registrado:", equipos_planta[0]["tag"], "-", equipos_planta[0]["nombre"])
print("Vibración actual de TUR-01:", equipos_planta[1]["vib_mms"], "mm/s")

Primer equipo registrado: CAL-01 - Caldera Principal
Vibración actual de TUR-01: 4.4 mm/s


## Paso 3 — Bucles `for`: Auditar Toda la Planta en 6 Líneas
En lugar de escribir cuatro bloques `if` repetidos, un bucle `for` recorre cada equipo de `equipos_planta` uno por uno y acumula estadísticas.

In [3]:
# 3. Auditoría automática de vibración con bucle 'for'
equipos_en_alerta = 0

print("=== REPORTE AUTOMÁTICO DE TURNO ===")
for eq in equipos_planta:
    margen = eq["limite_vib"] - eq["vib_mms"]
    if eq["vib_mms"] >= eq["limite_vib"]:
        equipos_en_alerta += 1
        estado = f"ALERTA (+{-margen:.1f} mm/s sobre límite)"
    else:
        estado = f"OK (margen {margen:.1f} mm/s)"
    print(f"[{eq['tag']}] {eq['nombre']:22s} | Vib: {eq['vib_mms']:.1f} mm/s -> {estado}")

print(f"\nResumen: {equipos_en_alerta} de {len(equipos_planta)} equipos requieren inspección.")

=== REPORTE AUTOMÁTICO DE TURNO ===
[CAL-01] Caldera Principal      | Vib: 2.1 mm/s -> OK (margen 1.4 mm/s)
[TUR-01] Turbina de Vapor       | Vib: 4.4 mm/s -> ALERTA (+0.4 mm/s sobre límite)
[BOM-02] Bomba de Alimentación  | Vib: 2.6 mm/s -> OK (margen 0.9 mm/s)
[COM-01] Compresor de Aire      | Vib: 4.7 mm/s -> ALERTA (+0.5 mm/s sobre límite)

Resumen: 2 de 4 equipos requieren inspección.


## Paso 4 — Funciones `def`: Empaqueta Tu Lógica de Ingeniería
Una función es una receta reutilizable: recibe entradas (parámetros), ejecuta un cálculo o validación y devuelve un resultado con `return`.

In [4]:
# 4. Definir funciones de ingeniería reutilizables
def evaluar_alarma_sensor(valor, limite_adv, limite_crit, unidad="mm/s"):
    """Clasifica una lectura industrial en Normal, Advertencia o Crítico."""
    if valor >= limite_crit:
        return f"CRÍTICO ({valor:.1f} {unidad} >= {limite_crit:.1f})"
    elif valor >= limite_adv:
        return f"ADVERTENCIA ({valor:.1f} {unidad} >= {limite_adv:.1f})"
    return f"NORMAL ({valor:.1f} {unidad})"

def calcular_eficiencia_pct(potencia_util_kw, potencia_entrada_kw):
    """Calcula eficiencia energética (%) protegiendo división por cero."""
    if potencia_entrada_kw <= 0:
        return 0.0
    return (potencia_util_kw / potencia_entrada_kw) * 100.0

print("Estado vibración BOM-02:", evaluar_alarma_sensor(2.6, 3.2, 4.5))
print("Estado vibración COM-01:", evaluar_alarma_sensor(4.7, 3.2, 4.5))
print("Estado temperatura CAL-01:", evaluar_alarma_sensor(196.0, 190.0, 200.0, unidad="°C"))
print(f"Eficiencia de bomba: {calcular_eficiencia_pct(36.4, 45.5):.1f}%")

Estado vibración BOM-02: NORMAL (2.6 mm/s)
Estado vibración COM-01: CRÍTICO (4.7 mm/s >= 4.5)
Estado temperatura CAL-01: ADVERTENCIA (196.0 °C >= 190.0)
Eficiencia de bomba: 80.0%


## Paso 5 — Intro a NumPy: Operaciones Vectorizadas Sin Bucles
En Python puro, convertir 10,000 lecturas requiere un bucle `for`. Con **NumPy** (`np.array`), multiplicas o sumas todo el vector de un solo golpe, hasta **50 veces más rápido**.

In [5]:
# 5. NumPy vectorizado para convertir y filtrar telemetría de caldera
import numpy as np

arr_c = np.array(temps_caldera)

# Conversión vectorizada de TODAS las lecturas a Fahrenheit y Kelvin en 1 línea
arr_f = arr_c * 9.0 / 5.0 + 32.0
arr_k = arr_c + 273.15

# Estadísticas y filtrado booleano inmediato
p90 = np.percentile(arr_c, 90)
lecturas_altas = arr_c[arr_c >= 190.0]

print("Arreglo en °C :", arr_c)
print("Arreglo en °F :", np.round(arr_f, 1))
print(f"Media: {arr_c.mean():.2f} °C | Desv. Est.: {arr_c.std():.2f} °C | P90: {p90:.2f} °C")
print("Lecturas >= 190 °C:", lecturas_altas, f"({len(lecturas_altas)} horas)")

Arreglo en °C : [181.2 183.5 185.  188.4 192.1 194.8 190.2 186.7]
Arreglo en °F : [358.2 362.3 365.  371.1 377.8 382.6 374.4 368.1]
Media: 187.74 °C | Desv. Est.: 4.25 °C | P90: 192.91 °C
Lecturas >= 190 °C: [192.1 194.8 190.2] (3 horas)


## Paso 6 — Error Lab 03: `IndexError` y `KeyError`
- **`IndexError`**: intentas acceder a una posición que no existe en la lista (recuerda que Python empieza a contar en `0`).
- **`KeyError`**: pides un nombre de clave inexistente en un diccionario. Solución segura: usar `.get(clave, valor_por_defecto)`.

In [6]:
# 6. Error Lab 03: IndexError en listas y KeyError en diccionarios
print("--- Demo 1: IndexError ---")
try:
    print(temps_caldera[20])
except IndexError as err:
    print("Error capturado -> IndexError:", err)
    print(f"Índices válidos van de 0 a {len(temps_caldera) - 1}. Último valor: {temps_caldera[-1]}")

print("\n--- Demo 2: KeyError y acceso seguro con .get() ---")
ficha = equipos_planta[0]
try:
    print(ficha["caudal_m3h"])
except KeyError as err:
    print("Error capturado -> KeyError para clave:", err)
    print("Acceso seguro con .get():", ficha.get("caudal_m3h", "No medido en este activo"))

--- Demo 1: IndexError ---
Error capturado -> IndexError: list index out of range
Índices válidos van de 0 a 7. Último valor: 186.7

--- Demo 2: KeyError y acceso seguro con .get() ---
Error capturado -> KeyError para clave: 'caudal_m3h'
Acceso seguro con .get(): No medido en este activo


## Paso 7 — Tu Turno (Ejercicios Módulo 3)
1. **Reto 3.1:** Crea una función `convertir_bar_a_psi(presion_bar)` que retorne `presion_bar * 14.5038`.
2. **Reto 3.2:** Aplica la función mediante un bucle `for` Y también usando `np.array` sobre la lista `[10.5, 12.1, 14.8, 11.9]` bar.

In [7]:
# Soluciones verificadas — Tu Turno Módulo 3
def convertir_bar_a_psi(presion_bar):
    return presion_bar * 14.5038

presiones_bar = [10.5, 12.1, 14.8, 11.9]

# Método 1: bucle for
presiones_psi_for = []
for p in presiones_bar:
    presiones_psi_for.append(round(convertir_bar_a_psi(p), 1))

# Método 2: NumPy vectorizado
presiones_psi_np = np.round(np.array(presiones_bar) * 14.5038, 1)

print("Reto 3.2 (for)  :", presiones_psi_for)
print("Reto 3.2 (NumPy):", presiones_psi_np)

Reto 3.2 (for)  : [152.3, 175.5, 214.7, 172.6]
Reto 3.2 (NumPy): [152.3 175.5 214.7 172.6]
